Import all the required libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

Load the dataset

In [2]:
X, y = load_iris(return_X_y=True)
X

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2],
       [5.4, 3.9, 1.7, 0.4],
       [4.6, 3.4, 1.4, 0.3],
       [5. , 3.4, 1.5, 0.2],
       [4.4, 2.9, 1.4, 0.2],
       [4.9, 3.1, 1.5, 0.1],
       [5.4, 3.7, 1.5, 0.2],
       [4.8, 3.4, 1.6, 0.2],
       [4.8, 3. , 1.4, 0.1],
       [4.3, 3. , 1.1, 0.1],
       [5.8, 4. , 1.2, 0.2],
       [5.7, 4.4, 1.5, 0.4],
       [5.4, 3.9, 1.3, 0.4],
       [5.1, 3.5, 1.4, 0.3],
       [5.7, 3.8, 1.7, 0.3],
       [5.1, 3.8, 1.5, 0.3],
       [5.4, 3.4, 1.7, 0.2],
       [5.1, 3.7, 1.5, 0.4],
       [4.6, 3.6, 1. , 0.2],
       [5.1, 3.3, 1.7, 0.5],
       [4.8, 3.4, 1.9, 0.2],
       [5. , 3. , 1.6, 0.2],
       [5. , 3.4, 1.6, 0.4],
       [5.2, 3.5, 1.5, 0.2],
       [5.2, 3.4, 1.4, 0.2],
       [4.7, 3.2, 1.6, 0.2],
       [4.8, 3.1, 1.6, 0.2],
       [5.4, 3.4, 1.5, 0.4],
       [5.2, 4.1, 1.5, 0.1],
       [5.5, 4.2, 1.4, 0.2],
       [4.9, 3

Data Splitting

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=44)

Scale the Features

In [4]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Hyperparameter Tuning

GridSearchCV

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'p': [1, 2]
}

grid_search = GridSearchCV(
    estimator = KNeighborsClassifier(),
    param_grid = param_grid,
    scoring = "accuracy",
    n_jobs = -1
)

grid_search.fit(X_train_scaled, y_train)

GridSearchCV(estimator=KNeighborsClassifier(), n_jobs=-1,
             param_grid={'n_neighbors': [3, 5, 7, 9, 11], 'p': [1, 2]},
             scoring='accuracy')

In [6]:
grid_search.best_params_

{'n_neighbors': 7, 'p': 2}

In [7]:
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_scaled, y_train)

KNeighborsClassifier(n_neighbors=7)

In [8]:
y_pred = knn.predict(X_test_scaled)
print(y_pred)

[2 0 1 1 2 0 2 2 2 1 0 1 0 2 0 0 2 1 0 2 1 2 2 1 2 1 0 1 0 1]


In [9]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix
accuracy_score(y_test, y_pred)

0.9666666666666667

In [10]:
confusion_matrix(y_test, y_pred)

array([[ 9,  0,  0],
       [ 0,  9,  0],
       [ 0,  1, 11]])

Tune only K via loop

In [11]:
from sklearn.model_selection import cross_val_score

k_values = range(1, 11)
scores = []

for k in k_values:
  knn = KNeighborsClassifier(n_neighbors=k)

  score = cross_val_score(knn, X_train_scaled, y_train, cv=5).mean()
  scores.append(score)

best_k = k_values[scores.index(max(scores))]
print(best_k)

7


KNN Imputation

In [12]:
df = pd.DataFrame({
    'age': [23, 25, np.nan, 35, 40],
    'salary': [50000, np.nan, 60000, 70000, 90000],
    'experience': [1, 2, 3, np.nan, 9]
})

df

,age,salary,experience
0,23.0,50000.0,1.0
1,25.0,NaN,2.0
2,NaN,60000.0,3.0
3,35.0,70000.0,NaN
4,40.0,90000.0,9.0


Apply KNN Imputer

In [13]:
from sklearn.impute import KNNImputer

imputer = KNNImputer(n_neighbors=3)
imputed_data = imputer.fit_transform(df)

df_imputed = pd.DataFrame(imputed_data, columns=df.columns)
df_imputed

,age,salary,experience
0,23.000000,50000.0,1.000000
1,25.000000,60000.0,2.000000
2,27.666667,60000.0,3.000000
3,35.000000,70000.0,4.666667
4,40.000000,90000.0,9.000000
